In [ ]:
#@title CMAscan functions

def is_valid_fasta_sequence(sequence):
    """Return True if the sequence contains only allowed amino acid symbols."""

    valid_chars = set("RHKDESTNQCGPAVILMFYWUOX")  # Valid characters in a protein sequence
    return all(aa in valid_chars for aa in sequence)


def process_fasta_file(file_name, file_content):
    """Read a FASTA file and stop if any sequence contains invalid symbols."""

    lines = file_content.decode().splitlines()
    header = None
    sequence = ""

    for line in lines:
        if line.startswith(">"):
            # New sequence header found
            if header is not None and sequence:
                # Validate the previous sequence before moving to the next one
                if not is_valid_fasta_sequence(sequence):
                    print(f"Invalid sequence in {file_name}: {header}")
                    sys.exit(1)  # Terminate the code execution with a non-zero status
            header = line[1:]  # Remove ">" from the header
            sequence = ""
        else:
            sequence += line.strip()

    # Validate the last sequence in the file
    if header is not None and sequence:
        if not is_valid_fasta_sequence(sequence):
            print(f"Invalid sequence in {file_name}: {header}")
            sys.exit(1)  # Terminate the code execution with a non-zero status


def is_canonical(motif):
    """Check whether a 5 aa sequence fits the canonical CMA motif rules."""
    if len(motif) != 5:
        return False

    side = ["Q", "N"]
    positive = ["K", "R"]
    negative = ["D", "E"]
    hydrophobic = ["F", "L", "I", "V"]

    # Rule 1: Only allowed amino acids
    if any(residue not in side + positive + negative + hydrophobic for residue in motif):
        return False

    # Rule 2: Starts or ends with Q or N
    if not (motif[0] in side or motif[-1] in side):
        return False

    # Rule 3: Exactly one Q or N
    if len(re.findall(f"[{''.join(side)}]", motif)) != 1:
        return False

    # Rule 4: One or two K or R
    if len(re.findall(f"[{''.join(positive)}]", motif)) not in [1, 2]:
        return False

    # Rule 5: Exactly one D or E
    if len(re.findall(f"[{''.join(negative)}]", motif)) != 1:
        return False

    # Rule 6: One or two F, L, I, or V
    if len(re.findall(f"[{''.join(hydrophobic)}]", motif)) not in [1, 2]:
        return False

    return True


def is_phosphorylated(motif):
    """
    Check if a given 5-aa sequence meets the criteria for a phosphorylated CMA (KFERQ-like) motif.

    Criteria:
    - Must contain only allowed residues (Q, N, K, R, T, Y, S, F, L, I, V)
    - Must start or end with Q or N
    - Must contain exactly one Q or N
    - Must contain 1–2 K or R residues
    - Must contain exactly one phosphorylatable residue (T, Y, or S)
    - Must contain 1–2 F, L, I, or V residues

    Args:
        motif (str): A 5-amino-acid subsequence.

    Returns:
        bool: True if motif is a phosphorylated CMA motif, False otherwise.
    """
    if len(motif) != 5:
        return False

    flank = ["Q", "N"]
    positive = ["K", "R"]
    phospho = ["T", "Y", "S"]
    hydrophobic = ["F", "L", "I", "V"]

    # Rule 1: Only allowed amino acids
    if any(residue not in flank + positive + phospho + hydrophobic for residue in motif):
        return False

    # Rule 2: Starts or ends with Q or N
    if not (motif[0] in flank or motif[-1] in flank):
        return False

    # Rule 3: Exactly one Q or N
    if len(re.findall(f"[{''.join(flank)}]", motif)) != 1:
        return False

    # Rule 4: One or two K or R
    if len(re.findall(f"[{''.join(positive)}]", motif)) not in [1, 2]:
        return False

    # Rule 5: Exactly one phosphorylatable residue (T, Y, S)
    if len(re.findall(f"[{''.join(phospho)}]", motif)) != 1:
        return False

    # Rule 6: One or two F, L, I, or V
    if len(re.findall(f"[{''.join(hydrophobic)}]", motif)) not in [1, 2]:
        return False

    return True


def find_pos_phos(motif, start):
    '''Find the position of the phosphorylated residue in a given motif.

    This function iterates through the motif sequence to check for the presence
    of any phosphorylated residues (amino acids: T, Y, or S). If a phosphorylated
    residue is found, it calculates the position of that residue in the original
    protein sequence using the `start` parameter and returns a tuple containing
    the phosphorylated residue and its position.

    Parameters:
        motif (str): The motif sequence to search for the phosphorylated residue.
        start (int): The start poskition of the motif in the original protein sequence.

    Returns:
        tuple: A tuple containing the phosphorylated residue and its position in
        the original protein sequence. If no phosphorylated residue is found in
        the motif, it returns `(None, None)`.

    Note:
        The function assumes that the `motif` sequence contains at most one phosphorylated
        residue (T, Y, or S). If multiple phosphorylated residues are present, the function
        will only return the first one it encounters.
    '''
    # List of negative (phosphorylated) residues
    negative_residues = ["T", "Y", "S"]

    for position, residue in enumerate(motif):
        if residue in negative_residues:
            # Calculate the position of the phosphorylated residue in the original sequence
            position_in_sequence = start + position + 1
            return residue, position_in_sequence

    # Return None if no phosphorylated residue is found
    return None, None


def is_acetylated(motif):
    """
    Check if a given 5-aa sequence meets the criteria for an acetylated CMA (KFERQ-like) motif.

    Criteria:
    - Must contain only allowed residues (K, R, D, E, F, L, I, V)
    - Must start or end with K
    - Must contain 1–3 K or R residues
    - Must contain exactly one D or E
    - Must contain 1–2 F, L, I, or V residues

    Args:
        motif (str): A 5-amino-acid subsequence.

    Returns:
        bool: True if motif is an acetylated CMA motif, False otherwise.
    """
    if len(motif) != 5:
        return False

    flank = ["K"]  # Acetylation-mimicking flanking residue
    positive = ["K", "R"]
    negative = ["D", "E"]
    hydrophobic = ["F", "L", "I", "V"]

    # Rule 1: Only allowed amino acids
    if any(residue not in flank + positive + negative + hydrophobic for residue in motif):
        return False

    # Rule 2: Starts or ends with K
    if not (motif[0] in flank or motif[-1] in flank):
        return False

    # Rule 3: One to three K or R
    if len(re.findall(f"[{''.join(positive)}]", motif)) not in [1, 2, 3]:
        return False

    # Rule 4: Exactly one D or E
    if len(re.findall(f"[{''.join(negative)}]", motif)) != 1:
        return False

    # Rule 5: One or two F, L, I, or V
    if len(re.findall(f"[{''.join(hydrophobic)}]", motif)) not in [1, 2]:
        return False

    return True


def find_pos_acetylo(motif, start):
    '''Find the position(s) of the acetylated residue (lysine) in a given motif.

    This function iterates through the motif sequence to check for the presence
    of lysine residues (amino acid 'K') at positions 1 and 5 (0-indexed) in the
    motif. If lysine residues are found at either of these positions, it calculates
    the position(s) of those lysine residues in the original protein sequence using
    the `start` parameter and returns a list of positions.

    Parameters:
        motif (str): The motif sequence to search for lysine residues.
        start (int): The start position of the motif in the original protein sequence.

    Returns:
        list: A list containing the position(s) of lysine residues in the original
        protein sequence. If no lysine residues are found on the flanks (positions 1 and 5)
        of the motif, it returns an empty list.

    Note:
        The positions in the returned list are 1-indexed, representing the positions of the
        lysine residues in the original protein sequence.
    '''
    # List to store positions of lysine residues (K) on the flanks (positions 1 and 5)
    flank_positions = []

    # Iterate through the motif to find lysine residues at positions 1 and 5
    for pos, res in enumerate(motif):
        if res == 'K':
            if pos == 0 or pos == 4:
                flank_positions.append(pos)

    if len(flank_positions) > 1:
        # If there are multiple lysine residues, calculate their positions and return as a list
        return [position + start + 1 for position in flank_positions]
    elif len(flank_positions) == 1:
        # If there is only one lysine residue, calculate its position and return as a single-element list
        return [flank_positions[0] + start + 1]
    else:
        # If no lysine residues are found on flanks, print an error message and return an empty list
        print('Error: Motif', motif, 'has no lysines on flanks')
        return []


def charge_transformation(seq):
    '''Transforms a given motif sequence into a sequence of amino acid categories based on their charge.

    This function takes a motif sequence as input and transforms each amino acid
    in the sequence into a category based on its charge property. The following
    categories are used:
    - '+' for positively charged amino acids (K, R)
    - '-' for negatively charged amino acids (D, E)
    - '*' for phosphorylated amino acids (T, Y, S)
    - '!' for amino acids on the side chain (Q, N)
    - '^' for hydrophobic amino acids (F, L, I, V)
    - For any other amino acids, it retains the original one-letter code.

    Parameters:
        seq (str): The motif sequence to transform.

    Returns:
        str: The transformed sequence with amino acid categories.

    Note:
        The function does not perform any validation on the input sequence. It assumes
        that the input sequence contains valid amino acid characters. Any non-standard
        characters in the input sequence will be treated as regular amino acids in the
        output.

        The transformed sequence will have the same length as the input sequence, and each
        character in the output represents the category of the corresponding amino acid
        in the input sequence.
    '''

    side = ["Q", "N"]
    positive = ["K", "R"]
    negative = ["D", "E"]
    phosphorylated = ["T", "Y", "S"]
    hydrophobic = ["F", "L", "I", "V"]

    # Initialize an empty string to store the transformed sequence
    seq_charge = ''

    # Iterate through the input sequence and transform each amino acid based on its charge property
    for aa in seq:
        if aa in side:
            seq_charge = seq_charge + '!'
        elif aa in positive:
            seq_charge = seq_charge + '+'
        elif aa in negative:
            seq_charge = seq_charge + '-'
        elif aa in hydrophobic:
            seq_charge = seq_charge + '^'
        elif aa in phosphorylated:
            seq_charge = seq_charge + '*'
        else:
            seq_charge = seq_charge + 'x'

    return seq_charge


def find_unique_k_mers(seq):
    '''Finds unique k-mers in a given sequence.

    This function takes a sequence as input and identifies all unique k-mers
    (subsequences of length k) present in the sequence. It returns a list of
    unique k-mers found in the sequence.

    Parameters:
        seq (str): The input sequence to search for k-mers.

    Returns:
        list: A list of unique k-mers found in the sequence.

    Note:
        The function was created to address the issue of different motif lengths
        used in scientific literature. Some authors might provide 6-letter KFERQ
        motifs, while others use the standard 5-letter KFERQ motif. To ensure
        consistency in motif comparison, this function extracts two unique 5-letter
        motifs from the 6-letter motif. For example, if the motif in a paper
        (and therefore in the Database) is "QKFERQ," the function will return
        two unique 5-letter motifs for comparison: "QKFER" and "KFERQ."

        The function assumes that the input sequence is a string containing valid
        characters. It does not perform any validation on the input.

        The function uses a sliding window approach to extract all possible k-mers
        from the input sequence and then filters out duplicate k-mers to return
        only the unique ones.

        The value of k is set to 5 by default, but you can modify it in the code
        if you want to find k-mers of a different length.

        The function returns the unique k-mers as a list of strings, and the order
        of appearance is preserved in the output list.
    '''

    k = 5  # Length of k-mers
    unique_k_mers = []  # List to store unique k-mers

    # Calculate the number of possible k-mers in the sequence
    n_mers = len(seq) - k + 1

    # Iterate through the sequence and find all possible k-mers
    for i in range(n_mers):
        mer = seq[i:i + k]

        # Check if the k-mer is not already present in the list
        if mer not in unique_k_mers:
            unique_k_mers.append(mer)

    return unique_k_mers


def compare_sequences(sequence, record_description, output_df, similarity_type):
    '''Compares the input sequence with the CMAscan Database.

    This function compares the given input sequence with the CMAscan Database
    using either an 'identical' or 'similar' comparison. It searches for motifs
    in the input sequence that match or are similar to motifs in the database
    and stores the results in the output DataFrame.

    Parameters:
        sequence (str): The input sequence to be compared with the database.
        record_description (str): Description or identifier of the input sequence.
        output_df (pandas.DataFrame): DataFrame to store the comparison results.
        similarity_type (str): Type of comparison to perform. It can be either
                                'identical' or 'similar'.

    Returns:
        pandas.DataFrame: The output DataFrame containing the comparison results.

    Note:
        The function assumes that the input sequence and record_description are
        valid strings and that the output_df is a pandas DataFrame with appropriate
        columns to store the results.

        The function compares each motif in the CMAscan Database with the input
        sequence. For some motifs in the CMAscan Database that are described as 6-letter
        motifs in PubMed, the function first extracts two unique 5-letter motifs
        from them to ensure consistent comparison with the 5-letter motifs in the
        input sequence. If the motif is already 5 amino acids long, it is used directly
        for comparison.

        If the similarity_type is set to 'identical', the function checks for exact
        matches between the extracted motifs and subsequences in the input sequence.
        If an exact match is found, the function stores the comparison result in the
        output DataFrame with the motif category and other relevant information from
        the CMAscan Database.

        If the similarity_type is set to 'similar', the function converts both the
        extracted motifs and subsequences in the input sequence into sequences of
        amino acid categories based on their charges using the charge_transformation
        function. It then checks for matches between the transformed motifs and
        subsequences. If a match is found, the function stores the comparison result
        in the output DataFrame with the motif category and other relevant information
        from the CMAscan Database.

        The output DataFrame will contain columns for the record description, matched
        motif subsequence, position in the input sequence, similarity type ('identical'
        or 'similar'), and other information from the CMAscan Database.
    '''

    head_db = list(motif_db.columns)

    for ind in motif_db.index:
            motif_from_db = motif_db['Motif'][ind]

            # If the motif is 6 letters, find two unique 5-letter motifs from it
            if len(motif_from_db) == 6:
                motifs_to_compare = find_unique_k_mers(motif_from_db)
            else:
                # If the motif is already 5 letters, use it directly for comparison
                motifs_to_compare = [motif_from_db]

            # Compare each motif with subsequences in the input sequence
            for motif in motifs_to_compare:
                n_mers = len(sequence) - len(motif) + 1

                # Perform either 'identical' or 'similar' comparison
                for i in range(n_mers):
                    mer = sequence[i:i + len(motif)]

                    if similarity_type == 'identical':
                        if mer == motif or mer == motif[::-1]:
                            result = [record_description, mer, str(i + 1), 'identical'] + list(motif_db.loc[ind, head_db])
                            output_df.loc[len(output_df)] = result
                    elif similarity_type == 'similar':
                        if charge_transformation(mer) == charge_transformation(motif) or \
                                charge_transformation(mer) == charge_transformation(motif[::-1]):
                            result = [record_description, mer, str(i + 1), 'similar'] + list(motif_db.loc[ind, head_db])
                            output_df.loc[len(output_df)] = result

    return output_df


def calculate_pssm_matrix(sequences, alphabet, background_distribution, k=1):
    """
    Calculate the Position-Specific Scoring Matrix (PSSM) for a list of sequences.

    This function takes a list of sequences (motifs), an alphabet, and a background
    amino acid distribution to compute the PSSM. It applies Laplace smoothing to
    the Position Frequency Matrix (PFM) to handle zero counts and computes the PSSM
    based on the provided background distribution.

    Args:
        sequences (list of str): List of motif sequences.
        alphabet (Bio.Alphabet): The alphabet used for the motifs.
        background_distribution (dict): Background amino acid distribution.
        k: Laplace smoothing parameter, default=1

    Returns:
        Bio.motifs.Motif: The Position-Specific Scoring Matrix (PSSM).

    Raises:
        ValueError: If the list of sequences is empty or if an error occurs during
                    the calculation of the PSSM.
    """
    from Bio import motifs

    if not sequences:
        raise ValueError("The list of sequences/motifs is empty.")

    try:

        # Reorient sequences to start with Q or N
        sequences = [reorient_motif(motif) for motif in sequences]

        # Create Bio.motifs.Motif object
        motif_object = motifs.create([Seq(seq, alphabet) for seq in sequences],
                                     alphabet=alphabet)

        # Calculate Position Frequency Matrix (PFM)
        pfm = motif_object.counts

        # Number of motifs (observations)
        num_motifs = len(sequences)
        # Number of amino acids in the alphabet
        num_amino_acids = len(alphabet)

        # Performing Laplace smoothing for each aa in each position
        m = 0
        for aa in pfm:
          m += 1
        pseudocount = 0
        n = len(sequences)

        for position in range(len(sequences[0])):
          for aa in pfm:
            instances = pfm.__getitem__(aa)[position]
            pseudocount = (instances + k)/(n + k*m)
            pfm.__getitem__(aa)[position] += pseudocount

        # Create Position Propability Matrix
        ppm = pfm.normalize()

        # Calculate log_odds ratios for PSSM matrix
        pssm_matrix = ppm.log_odds(background_distribution)

        return pssm_matrix

    except Exception as e:
        raise ValueError(f"Error calculating the PSSM: {e}")


def calculate_pssm_charge_matrix(sequences, background_distribution, k=1):
    '''Calculate the Position-Specific Scoring Matrix (PSSM) from validated motifs in the CMAscan Database.

    This function loads the CMAscan Database, filters the motifs based on specific criteria,
    reorients the motifs to start with "Q" or "N" if needed, and then calculates the PSSM matrix
    from the filtered motifs. As a background it uses precalculated mean amino acid distributions from mammalian proteomes.

    Returns:
        Bio.motifs.Motif: The Position-Specific Scoring Matrix (PSSM).

    Raises:
        ValueError: If the CMAscan Database cannot be accessed or loaded.
    '''
    from Bio import motifs

    if not sequences:
        raise ValueError("The list of sequences/motifs is empty.")

    try:

        # Reorient sequences to start with Q or N
        sequences = [reorient_motif(motif) for motif in sequences]

        # Perform charge transformation
        sequences = [Seq(charge_transformation(motif)) for motif in sequences]

        # Number of observations (motifs)
        n = len(sequences)

        sequences = motifs.create(sequences, alphabet="!+-*^x")

        aa_categories = {
            "!": ["Q", "N"],
            "+": ["K", "R"],
            "-": ["D", "E"],
            "*": ["T", "Y", "S"],
            "^": ["F", "L", "I", "V"],
            "x" : ['A', 'C', 'G', 'H', 'M', 'P', 'W', 'X', 'Z', 'B']
        }

        background_categories = {category: 0 for category in aa_categories}

        for amino_acid, value in background_distribution.items():
            for category, aa_list in aa_categories.items():
                if amino_acid in aa_list:
                    background_categories[category] += value

        # Create Position Frequency Matrix
        pfm = sequences.counts

        # Performing Laplace smoothing for each aa in each position
        m = 0
        for aa in pfm:
          m += 1
        pseudocount = 0

        for position in range(5):
          for aa in pfm:
            instances = pfm.__getitem__(aa)[position]
            pseudocount = (instances + k)/(n + k*m)
            pfm.__getitem__(aa)[position] += pseudocount

        # Create Position Propability Matrix
        ppm = pfm.normalize()

        # Calculate log_odds ratios for PSSM matrix
        pssm_matrix = ppm.log_odds(background_categories)

        return pssm_matrix

    except Exception as e:
        raise ValueError(f"Error accessing or loading the CMAscan Database: {e}")


def calculate_pssm_matrix_expanded(matrix_alphabet):
    '''Calculate the Position-Specific Scoring Matrix (PSSM) from validated motifs in the CMAscan Database.

    This function loads the CMAscan Database, filters the motifs based on specific criteria,
    reorients the motifs to start with "Q" or "N" if needed, and then calculates the PSSM matrix
    from the filtered motifs. As a background it uses precalculated mean amino acid distributions from mammalian proteomes.

    Returns:
        Bio.motifs.Motif: The Position-Specific Scoring Matrix (PSSM).

    Raises:
        ValueError: If the CMAscan Database cannot be accessed or loaded.
    '''

    human_aa_distribution = {
        'S': 0.0833640853959927,
        'T': 0.0534707922749892,
        'D': 0.0473298742499409,
        'Q': 0.0469573385466771,
        'Y': 0.0268749552304211,
        'F': 0.0373789229889875,
        'G': 0.0654433309782482,
        'P': 0.0623692840533557,
        'R': 0.0566692634635304,
        'L': 0.100605377772131,
        'V': 0.0605991043212125,
        'M': 0.0216626159993309,
        'C': 0.0231585862417482,
        'H': 0.0260597646198473,
        'E': 0.0694340464165267,
        'N': 0.0356027235615439,
        'A': 0.0699340171989205,
        'K': 0.0567922470525259,
        'W': 0.012267877287158,
        'I': 0.0440021611837513,
        'U': 2.5774934115491795e-06,
        'X': 2.07837023259903e-05,
        'Z': 7.978272463796199e-07,
        'B': 7.669243924527316e-07
        }

    try:
        # Load the motif database from the CMAscan repository
        motif_db = pd.read_csv(resolve_repo_file('dataset/cma_motif_dataset.csv'), delimiter=',', encoding='unicode_escape')

        # Filter motifs based on the current dataset codes (Type = 'C', MDM = 'Yes', Biological process = 'CMA')
        filtered_db = motif_db[
            (motif_db['Type'] == TYPE_CANONICAL) &
            (motif_db['MDM'] == MDM_CONFIRMED) &
            (motif_db['Biological process'] == 'CMA')
        ]

        # Reorient motifs to start with "Q" or "N" if needed
        for index, row in filtered_db.iterrows():
            filtered_db.loc[index, 'Motif'] = reorient_motif(row['Motif'])

        # Collect motifs and create a Bio.motifs.Motif object
        filtered_motifs = [Seq(motif) for motif in filtered_db['Motif']]

        new_kferqs = [
                      'NVRFD',
                      'QEILR',
                      'QEILK',
                      'QELFR',
                      'QIERF'
                  ]
        for motif in new_kferqs:
          filtered_motifs.append(motif)

        # Number of observations (motifs)
        n = len(filtered_motifs)

        filtered_motifs = motifs.create(filtered_motifs, matrix_alphabet)

        # Create Position Frequency Matrix
        pfm = filtered_motifs.counts

        # Performing Laplace smoothing for each aa in each position
        m = 0
        for aa in pfm:
          m += 1
        pseudocount = 0
        k = 0.1

        for position in range(5):
          for aa in pfm:
            instances = pfm.__getitem__(aa)[position]
            pseudocount = (instances + k)/(n + k*m)
            pfm.__getitem__(aa)[position] += pseudocount

        # Create Position Propability Matrix
        ppm = pfm.normalize()

        # Calculate log_odds ratios for PSSM matrix
        background = human_aa_distribution
        pssm_matrix = ppm.log_odds(background)

        return pssm_matrix

    except Exception as e:
        raise ValueError(f"Error accessing or loading the CMAscan Database: {e}")


def calculate_pssm_charge_matrix_expanded():
    '''Calculate the Position-Specific Scoring Matrix (PSSM) from validated motifs in the CMAscan Database.

    This function loads the CMAscan Database, filters the motifs based on specific criteria,
    reorients the motifs to start with "Q" or "N" if needed, and then calculates the PSSM matrix
    from the filtered motifs. As a background it uses precalculated mean amino acid distributions from mammalian proteomes.

    Returns:
        Bio.motifs.Motif: The Position-Specific Scoring Matrix (PSSM).

    Raises:
        ValueError: If the CMAscan Database cannot be accessed or loaded.
    '''

    human_aa_distribution = {
        'S': 0.0833640853959927,
        'T': 0.0534707922749892,
        'D': 0.0473298742499409,
        'Q': 0.0469573385466771,
        'Y': 0.0268749552304211,
        'F': 0.0373789229889875,
        'G': 0.0654433309782482,
        'P': 0.0623692840533557,
        'R': 0.0566692634635304,
        'L': 0.100605377772131,
        'V': 0.0605991043212125,
        'M': 0.0216626159993309,
        'C': 0.0231585862417482,
        'H': 0.0260597646198473,
        'E': 0.0694340464165267,
        'N': 0.0356027235615439,
        'A': 0.0699340171989205,
        'K': 0.0567922470525259,
        'W': 0.012267877287158,
        'I': 0.0440021611837513,
        'U': 2.5774934115491795e-06,
        'X': 2.07837023259903e-05,
        'Z': 7.978272463796199e-07,
        'B': 7.669243924527316e-07
        }


    try:
        # Load the motif database from the CMAscan repository
        motif_db = pd.read_csv(resolve_repo_file('dataset/cma_motif_dataset.csv'), delimiter=',', encoding='unicode_escape')

        # Filter motifs based on the current dataset codes (Type = 'C', MDM = 'Yes', Biological process = 'CMA')
        filtered_db = motif_db[
            (motif_db['Type'] == TYPE_CANONICAL) &
            (motif_db['MDM'] == MDM_CONFIRMED) &
            (motif_db['Biological process'] == 'CMA')
        ]

        # Reorient motifs to start with "Q" or "N" if needed
        for index, row in filtered_db.iterrows():
            filtered_db.loc[index, 'Motif'] = reorient_motif(row['Motif'])

        # Collect motifs and create a Bio.motifs.Motif object
        filtered_motifs = [Seq(charge_transformation(motif)) for motif in filtered_db['Motif']]

        new_kferqs = [
                      'NVRFD',
                      'QEILR',
                      'QEILK',
                      'QELFR',
                      'QIERF'
                  ]
        for motif in new_kferqs:
          filtered_motifs.append(Seq(charge_transformation(motif)))

        # Number of observations (motifs)
        n = len(filtered_motifs)

        filtered_motifs = motifs.create(filtered_motifs, alphabet="!+-*^x")

        aa_categories = {
            "!": ["Q", "N"],
            "+": ["K", "R"],
            "-": ["D", "E"],
            "*": ["T", "Y", "S"],
            "^": ["F", "L", "I", "V"],
            "x" : ['A', 'C', 'G', 'H', 'M', 'P', 'W', 'X', 'Z', 'B']
        }

        background_categories = {category: 0 for category in aa_categories}

        for amino_acid, value in human_aa_distribution.items():
            for category, aa_list in aa_categories.items():
                if amino_acid in aa_list:
                    background_categories[category] += value

        # Create Position Frequency Matrix
        pfm = filtered_motifs.counts

        # Performing Laplace smoothing for each aa in each position
        m = 0
        for aa in pfm:
          m += 1
        pseudocount = 0
        k = 0.1

        for position in range(5):
          for aa in pfm:
            instances = pfm.__getitem__(aa)[position]
            pseudocount = (instances + k)/(n + k*m)
            pfm.__getitem__(aa)[position] += pseudocount

        # Create Position Propability Matrix
        ppm = pfm.normalize()

        # Calculate log_odds ratios for PSSM matrix
        pssm_matrix = ppm.log_odds(background_categories)

        return pssm_matrix

    except Exception as e:
        raise ValueError(f"Error accessing or loading the CMAscan Database: {e}")


def reorient_motif(motif):
    '''
    Reorients a motif to start with "Q" or "N" if it doesn't.

    Parameters:
        motif (str): The input motif sequence.

    Returns:
        str: The reoriented motif sequence.
    '''

    # Check if the motif starts with "Q" or "N"
    if motif.isalpha() and motif[0] not in ["Q", "N"]:
        # If it doesn't, reverse the motif sequence
        return motif[::-1]
    elif "!" in motif and motif[0] not in ["!"]:
        return motif[::-1]

    # If it starts with "Q" or "N", return the original motif
    return motif


def calculate_pssm_score(motif, pssm_matrix):
    '''
    Calculate the PSSM score for a given motif using the PSSM matrix.

    This function calculates the Position-Specific Scoring Matrix (PSSM) score for a given motif
    based on the PSSM matrix. The PSSM score is the sum of the scores of individual amino acids
    at their respective positions in the motif, as defined by the PSSM matrix.

    Parameters:
        motif (str): The input motif sequence.
        pssm_matrix (Bio.motifs.Motif): The Position-Specific Scoring Matrix (PSSM).

    Returns:
        float: The calculated PSSM score.
    '''
    motif = reorient_motif(motif)
    # Initialize the PSSM score to zero
    pssm_score = 0
    motif = motif.replace('U', 'C')
    for i, aa in enumerate(motif):
        pssm_score += pssm_matrix.__getitem__(aa)[i]

    return pssm_score


def extract_frame(sequence, index, length):
    """
    Extracts a subsequence (frame) from a given sequence centered around a specified index.

    Args:
        sequence (str): The input sequence.
        index (int): The index around which the subsequence should be centered.
        length (int, optional): The desired length of the subsequence. Default is 1000.

    Returns:
        str: The extracted subsequence (frame).
    """
    # Ensure the length is an even number
    length = length // 2 * 2  # Make the length even

    # Calculate the left and right bounds of the substring
    left_bound = max(index - length // 2, 0)  # Calculate the start of the substring
    right_bound = min(index + length // 2, len(sequence))  # Calculate the end of the substring

    # Adjust bounds if sequence length is less than requested length
    if right_bound - left_bound < length:
        excess = length - (right_bound - left_bound)  # Calculate the amount of excess needed
        if left_bound == 0:
            right_bound = min(right_bound + excess, len(sequence))  # Adjust the right bound
        else:
            left_bound = max(left_bound - excess, 0)  # Adjust the left bound

    # Extract the frame
    frame = sequence[left_bound:right_bound]  # Extract the specified frame of the sequence

    #return frame  # Return the extracted frame
    return frame, left_bound  # Return the extracted frame and the start position of the frame


def musited(seq, mode='all'):

    if mode == 'all':
        model = "Phosphoserine_Phosphothreonine;Phosphotyrosine;N6-acetyllysine"
    elif mode == 'phosphorylation':
        model = "Phosphoserine_Phosphothreonine;Phosphotyrosine"
    elif mode == 'acetylation':
        model = 'N6-acetyllysine'
    else:
        print('Error: wrong mode.')

    results = {}

    # MusiteDeep API request
    url_base = "http://api.musite.net/musitedeep/" + model + "/"
    url = url_base + seq
    myResponse = requests.get(url)

    if myResponse.ok:
        jData = json.loads(myResponse.content.decode('utf-8'))
        if "Error" in jData.keys():
            print(jData["Error"])
        else:
            # Ordering data
            for i in range(len(jData['Results'])):
                residue = jData['Results'][i]['PTMscores']
                position = int(jData['Results'][i]['Residue'])
                PTMscore = jData['Results'][i]['Cutoff=0.5'].split(';')[0]
                # Adding PTMs to results
                results[int(position)] = PTMscore
    else:
      myResponse.raise_for_status()

    return results


def check_for_musitedeep(seq):

    if len(seq) > 999:
        return False
    else:
        return True


def print_progress(progress):
    timestamp = time.strftime("%Y-%m-%d %H:%M:%S", time.localtime())
    print(f"[{timestamp}] Progress: {progress:.2f}%")
    return f"[{timestamp}] Progress: {progress:.2f}%"


def run_iupred_analysis(sequence, mode, use_anchor=True):
    """Run IUPred3 analysis on a protein sequence with optional ANCHOR2 prediction.

    Args:
        sequence (str): Protein sequence to analyze.
        mode (str): "long", "short", or "glob".
        use_anchor (bool): If True, enables ANCHOR2 prediction.

    Returns:
        str: Raw output from IUPred3.
    """

    if mode not in {'long', 'short', 'glob'}:
        raise ValueError("Error: Wrong mode. Use 'long', 'short', or 'glob'.")

    if len(sequence) < 20:
        return [(i + 1, sequence[i], float('nan')) for i in range(len(sequence))]

    # Path to iupred3.py script
    iupred_script_path = os.path.join(folder_path, 'iupred3', 'iupred3.py')

    # Use a temporary file to store the FASTA sequence
    with tempfile.NamedTemporaryFile(mode="w+", delete=False, suffix=".fasta") as temp_fasta:
        temp_fasta.write(f">temp_sequence\n{sequence}\n")
        temp_fasta_path = temp_fasta.name

    # Build command
    command = ['python3', iupred_script_path, temp_fasta_path, mode]
    if use_anchor:
        command.append('-a')  # Enable ANCHOR2

    try:
        # Run IUPred3 with optional ANCHOR2
        result = subprocess.run(
            command,
            capture_output=True,
            text=True,
            check=True
        )
        output = result.stdout
    except subprocess.CalledProcessError as e:
        raise RuntimeError(f"Error running IUPred3: {e.stderr}")
    finally:
        os.remove(temp_fasta_path)  # Clean up temp file

    return output


def extract_iupred_scores(iupred_output):
    """Extract IUPred3 and ANCHOR2 scores from the output text.

    Args:
        iupred_output (str): The output text from IUPred3 analysis.

    Returns:
        tuple: Two lists of tuples (position, IUPred3 score) and (position, ANCHOR2 score).
    """

    iupred_scores = []
    anchor_scores = []

    lines = iupred_output.strip().split("\n")

    for line in lines:
        if line.startswith("#") or line.startswith("POS"):
            continue  # Skip headers

        parts = line.split()
        if len(parts) < 4:
            continue  # Skip malformed lines

        try:
            position = int(parts[0])  # Position
            iupred_score = float(parts[2])  # IUPred3 score
            anchor_score = float(parts[3])  # ANCHOR2 score

            iupred_scores.append((position, iupred_score))
            anchor_scores.append((position, anchor_score))

        except ValueError:
            continue  # Skip lines with unexpected formatting

    return iupred_scores, anchor_scores



def calculate_average_iupred_score(scores, i):
    """Calculate the average IUPRED score for amino acids from i+1 to i+5.

    Args:
        scores (list): List of tuples containing amino acid positions and their IUPRED scores.
        i (int): Position of the amino acid (0 indexed).

    Returns:
        float: Average IUPRED score for the specified range of amino acids.
    """
    if i < 0 or i >= len(scores) - 4:
        raise ValueError("Invalid position i")

    average_score = sum(score for _, score in scores[i+1:i+6]) / 5

    return average_score


def calculate_median_iupred_score(scores, i):
    """Calculate the median IUPRED score for amino acids from i+1 to i+5.

    Args:
        scores (list): List of tuples containing amino acid positions and their IUPRED scores.
        i (int): Position of the amino acid (0 indexed).

    Returns:
        float: Median IUPRED score for the specified range of amino acids.
    """
    if i < 0 or i >= len(scores) - 4:
        raise ValueError("Invalid position i")

    median_score = statistics.median(score for _, score in scores[i+1:i+6])

    return median_score


def pseudophosphorylate(motif, pseudophosphorylation='D'):
    """
    Pseudophosphorylation function replaces specific amino acid residues ('S', 'Y', 'T')
    in a given motif with the specified pseudophosphorylation amino acid ('E' or 'D').

    Args:
    motif (str): The input motif (sequence of amino acids).
    pseudophosphorylation (str, optional): The amino acid to replace 'S', 'Y', 'T' with.
        Should be 'D' (default) or 'E'.

    Returns:
    str: The pseudophosphorylated motif with 'S', 'Y', 'T' replaced by the specified amino acid.
    """
    # Define a list of amino acid residues to be replaced
    aa_to_replace = ['S', 'Y', 'T']

    # Check if the pseudophosphorylation parameter is valid ('E' or 'D')
    if pseudophosphorylation not in ['E', 'D']:
        raise ValueError("Invalid pseudophosphorylation amino acid. Should be 'E' or 'D'.")

    # Use a list comprehension to iterate through each amino acid in the motif
    # and replace 'S', 'Y', 'T' with the specified amino acid, while keeping other amino acids unchanged
    pseudophosphorylated_motif = ''.join([pseudophosphorylation if aa in aa_to_replace else aa for aa in motif])

    # Return the pseudophosphorylated motif
    return pseudophosphorylated_motif


def pseudoacetylate(motif):

    motif = 'Q' + motif[1:]

    return motif


def delete_commas_from_protein_descriptors(csv_file_path):
    """
    This function removes additional commas in a CSV file, which may be present due to
    commas in protein descriptors. These extra commas can affect the structure of a
    DataFrame and need to be deleted for further analysis.

    :param csv_file_path: The path to the CSV file to be processed.
    """
    try:
        # Open the CSV file for reading
        with open(csv_file_path, 'r') as file:
            # Read the first line (headers) and store it in headers variable
            headers = file.readline()

            # Count the number of commas in the headers
            commas_in_headers = headers.count(',')

            # Initialize a variable to store the polished content
            polished_content = headers

            # Iterate through the rest of the file content
            for line in file:
                # Count the number of commas in the current line
                commas_in_line = line.count(',')

                if commas_in_line > commas_in_headers:

                    # Calculate the number of extra commas to delete
                    extra_commas_to_delete = commas_in_line - commas_in_headers

                elif commas_in_line < commas_in_headers and commas_in_line > 1:
                    extra_commas_to_delete = commas_in_headers - 1

                else:
                    extra_commas_to_delete = 0

                # Remove the extra commas from the current line
                polished_line = line.replace(',', '', extra_commas_to_delete)

                # Append the polished line to the content
                polished_content += polished_line

        # Reopen the file in write mode to overwrite its content
        with open(csv_file_path, 'w') as file:
            # Write back the polished content, which includes headers
            file.write(polished_content)

        print(f"CSV file '{csv_file_path}' has been processed successfully.")

    except FileNotFoundError:
        print(f"File '{csv_file_path}' not found.")
    except Exception as e:
        print("An error occurred:", str(e))


def extract_substring_around_motif(input_string, motif, substring_length):

    # Find the index of the motif in the input string
    motif_index = input_string.find(motif)

    if motif_index == -1:
        # Motif not found in the input string
        return None

    # Calculate the start and end indices for the substring
    start_index = max(0, motif_index - substring_length)
    end_index = min(len(input_string), motif_index + len(motif) + substring_length)

    # Extract the substring
    extracted_substring = input_string[start_index:end_index]

    return extracted_substring


def is_potential_motif(mer):
    return is_canonical(mer) or is_phosphorylated(mer) or is_acetylated(mer)




In [ ]:
#@title Import libraries

import sys
!{sys.executable} -m pip install -q biopython openpyxl plotly seaborn requests pandas numpy matplotlib

from Bio import SeqIO, motifs
from Bio.Seq import Seq
from IPython.display import display
import re
import csv
import pandas as pd
import numpy as np
import seaborn as sns
import math
import statistics
import requests
import json
import subprocess
import tempfile
import os
import shutil
import time
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.patheffects as pe

REPO_OWNER = 'Marcin-Lubocki'
REPO_NAME = 'CMAscan'
RAW_BASE_URL_CANDIDATES = [
    f'https://raw.githubusercontent.com/{REPO_OWNER}/{REPO_NAME}/main',
    f'https://raw.githubusercontent.com/{REPO_OWNER}/{REPO_NAME}/master',
]

BASE_DIR = Path('/content') if Path('/content').exists() else Path.cwd()
REPO_ROOT = Path.cwd()
REQUEST_TIMEOUT = 120
FIGURE_WIDTH_PRESETS = {
    'full_width': 8.5,
    'two_column': 4.0,
}


def iter_repo_sources(relative_path):
    local_path = REPO_ROOT / relative_path
    if local_path.exists():
        yield str(local_path)
    for base_url in RAW_BASE_URL_CANDIDATES:
        yield f'{base_url}/{relative_path}'


def resolve_repo_file(relative_path):
    for source in iter_repo_sources(relative_path):
        if not str(source).startswith('http'):
            return str(source)
        try:
            response = requests.head(source, timeout=30, allow_redirects=True)
            if response.ok:
                return source
        except requests.RequestException:
            pass
    return f'{RAW_BASE_URL_CANDIDATES[0]}/{relative_path}'


def fetch_repo_file(relative_path, destination=None):
    target = Path(destination) if destination else Path(relative_path).name
    errors = []

    for source in iter_repo_sources(relative_path):
        try:
            if str(source).startswith('http'):
                response = requests.get(source, timeout=REQUEST_TIMEOUT)
                response.raise_for_status()
                target.write_bytes(response.content)
            else:
                shutil.copy(source, target)
            return str(target)
        except Exception as exc:
            errors.append(f'{source} -> {exc}')

    raise RuntimeError('Failed to fetch file: ' + '; '.join(errors))


def get_publication_figsize(default_width, default_height, width_mode='full_width', fig_height=None):
    if width_mode not in FIGURE_WIDTH_PRESETS:
        raise ValueError(f"Unsupported width_mode: {width_mode}. Use one of {tuple(FIGURE_WIDTH_PRESETS)}")

    fig_width = FIGURE_WIDTH_PRESETS[width_mode]
    if fig_height is None:
        fig_height = fig_width * (default_height / default_width)

    return fig_width, fig_height


def normalise_save_path(save_path, default_extension='svg'):
    if not save_path:
        return None

    save_path = str(save_path)
    root, ext = os.path.splitext(save_path)
    if ext:
        return root + '.' + default_extension
    return save_path + '.' + default_extension


folder_path = os.path.join(str(BASE_DIR), 'ExternalSoftware')
os.makedirs(folder_path, exist_ok=True)

iupred_script_path = Path(folder_path) / 'iupred3' / 'iupred3.py'
if not iupred_script_path.exists():
    iupred_archive = fetch_repo_file('ExternalSoftware/iupred3.tar.gz', 'iupred3.tar.gz')
    subprocess.run(['tar', '-xzf', iupred_archive, '-C', folder_path], check=True)



In [ ]:
#@title Load the CMAscan dataset

motif_db = pd.read_csv(resolve_repo_file('dataset/cma_motif_dataset.csv'), delimiter=',', encoding='unicode_escape')

TYPE_CANONICAL = "C"
TYPE_ATYPICAL = "A"

MDM_CONFIRMED = "Yes"
MDM_NOT_CONFIRMED = "No"
MDM_IN_SILICO = "In silico"

DATASET_POSITIVE = "P"
DATASET_PUTATIVE = "P*"
DATASET_POSITIVE_ATYPICAL = "PA"
DATASET_PUTATIVE_ATYPICAL = "PA*"
DATASET_EXCLUDED = "E"

NOT_INVESTIGATED = "NI"
INVESTIGATION_COLUMNS = [
    "Interaction with LAMP2A",
    "Interaction with HSC70",
    "Association with lysosomes",
]

TYPE_LABELS = {
    TYPE_CANONICAL: "canonical",
    TYPE_ATYPICAL: "atypical",
}

DATASET_LABELS = {
    DATASET_POSITIVE: "positive canonical",
    DATASET_PUTATIVE: "putative canonical",
    DATASET_POSITIVE_ATYPICAL: "positive atypical",
    DATASET_PUTATIVE_ATYPICAL: "putative atypical",
    DATASET_EXCLUDED: "excluded",
}

print(f"Rows loaded: {len(motif_db)}")
print("Type codes:", TYPE_LABELS)
print("Dataset codes:", DATASET_LABELS)
print("MDM values:", sorted(motif_db["MDM"].dropna().astype(str).unique().tolist()))



In [ ]:
#@title Load the dataset protein sequences
dataset_seqs_path = fetch_repo_file('dataset/dataset_seqs.fasta', 'dataset_seqs.fasta')
dataset_hq_seqs_path = fetch_repo_file('dataset/dataset_hq_seqs.fasta', 'dataset_hq_seqs.fasta')


In [ ]:
motif_db


In [ ]:
#@title Check UniProt IDs in FASTA files

from Bio import SeqIO
import pandas as pd

def get_motif_db_rows_with_missing_uniprot_ids(motif_db, fasta_path):
    """Return rows whose UniProt IDs are missing from the FASTA file."""
    # Extract UniProt IDs from FASTA
    fasta_ids = set()
    for record in SeqIO.parse(fasta_path, "fasta"):
        header_parts = record.id.split("|")
        if len(header_parts) >= 2:
            fasta_ids.add(header_parts[1])

    # Clean and split UniProt ID column
    df = motif_db.copy()
    df = df[~df["UniProt ID"].isna()]
    df["UniProt ID"] = df["UniProt ID"].astype(str).str.strip()
    df = df[~df["UniProt ID"].isin(["", "NA", "nan"])]

    # Determine which rows have all UniProt IDs missing
    def all_ids_missing(uniprot_str):
        ids = [x.strip() for x in uniprot_str.split(";")]
        return all(uid not in fasta_ids for uid in ids)

    missing_rows = df[df["UniProt ID"].apply(all_ids_missing)]

    if missing_rows.empty:
        print("All UniProt IDs from motif_db are present in the FASTA file.")
    else:
        print("Some UniProt IDs from motif_db are missing in the FASTA file.")

    return missing_rows

print("Check dataset_seqs.fasta:")
missing_rows_1 = get_motif_db_rows_with_missing_uniprot_ids(motif_db, "dataset_seqs.fasta")

print("\nCheck dataset_hq_seqs.fasta:")
missing_rows_2 = get_motif_db_rows_with_missing_uniprot_ids(motif_db, "dataset_hq_seqs.fasta")
print('Comment: Check if all of them are just non reference.')
missing_rows_2


In [ ]:
#@title Check motifs in FASTA files

from Bio import SeqIO
import pandas as pd

def verify_motifs_in_fasta_by_fasta(motif_db, fasta_path):
    """Check whether motifs from the dataset are present in the FASTA sequences."""
    # Load sequences from FASTA
    seq_dict = {}
    for record in SeqIO.parse(fasta_path, "fasta"):
        header_parts = record.id.split("|")
        if len(header_parts) >= 2:
            uniprot_id = header_parts[1]
            seq_dict[uniprot_id] = str(record.seq)

    # Clean motif_db: drop NAs and extract UniProt ID only
    df = motif_db.copy()
    df = df[~df["UniProt ID"].isna()]
    df["UniProt ID"] = df["UniProt ID"].astype(str).str.strip()
    df = df[~df["UniProt ID"].isin(["", "NA", "nan"])]
    df["UniProt Base"] = df["UniProt ID"].str.split(";").str[0]

    # Filter to only UniProt IDs present in the FASTA file
    df = df[df["UniProt Base"].isin(seq_dict.keys())]

    # Check each motif per protein
    failures = []
    for uniprot_id, seq in seq_dict.items():
        motifs = df[df["UniProt Base"] == uniprot_id]["Motif"].dropna().unique()
        for motif in motifs:
            motif = str(motif).strip()
            if motif not in seq:
                failures.append({
                    "UniProt ID": uniprot_id,
                    "Motif": motif,
                    "Issue": "Motif not found in sequence"
                })

    result_df = pd.DataFrame(failures)
    if result_df.empty:
        print("All motifs present in the proteins listed in this FASTA.")
    else:
        print("Missing motifs detected in FASTA. See DataFrame.")

    return result_df

print('Check dataset_seqs.fasta:')
motif_db_missing = verify_motifs_in_fasta_by_fasta(motif_db, "dataset_seqs.fasta")
print('Check dataset_hq_seqs.fasta:')
motif_db_hq_missing = verify_motifs_in_fasta_by_fasta(motif_db, "dataset_hq_seqs.fasta")


In [ ]:
#@title Basic report on the CMAscan dataset

from datetime import datetime
from collections import defaultdict

print(f'CMAscan dataset report [{datetime.now().strftime("%Y-%m-%d")}]')
print()
print(f"Studies included from {motif_db['Year'].min()} to {motif_db['Year'].max()}")
print("Type codes: C = canonical, A = atypical")
print("Dataset codes: P = positive canonical, P* = putative canonical, PA = positive atypical, PA* = putative atypical, E = excluded")
print("Investigation codes: NI = not investigated, CoLoc = co-localization, IsoLyso = isolated lysosome assay")
print()

unique_doi = []
for doi in motif_db['DOI'].dropna().astype(str):
    doi_clean = doi.rstrip('.')
    if doi_clean not in unique_doi:
        unique_doi.append(doi_clean)
print('Unique DOI numbers:', len(unique_doi))

unique_uniprot_ids = []
for uniprot_id in motif_db['UniProt ID']:
    if uniprot_id not in unique_uniprot_ids:
        unique_uniprot_ids.append(uniprot_id)
print('Unique UniProt IDs:', len(unique_uniprot_ids))

unique_motifs = []
for motif in motif_db['Motif']:
    if motif not in unique_motifs:
        unique_motifs.append(motif)
print('Unique motifs:', len(unique_motifs))
print('Motifs published as longer sequences than 5 aa:', len(motif_db[motif_db['Motif'].str.len() > 5]))
print('Motifs with MDM = Yes:', len(motif_db[motif_db['MDM'] == MDM_CONFIRMED]))
print()
print('Type counts: / MDM = Yes')
print()
for element in sorted(motif_db['Type'].dropna().unique()):
    label = TYPE_LABELS.get(element, element)
    total = len(motif_db[motif_db['Type'] == element])
    mdm_yes = len(motif_db[(motif_db['Type'] == element) & (motif_db['MDM'] == MDM_CONFIRMED)])
    print(f"{element} ({label}): {total} / {mdm_yes}")

print()
print('Dataset counts:')
for element in sorted(motif_db['Dataset'].dropna().unique()):
    label = DATASET_LABELS.get(element, element)
    print(f"{element} ({label}): {len(motif_db[motif_db['Dataset'] == element])}")

# Detect duplicated motifs after reorientation (based on protein names)
reoriented_map = defaultdict(list)
for idx, row in motif_db.iterrows():
    reoriented = reorient_motif(row['Motif'])
    reoriented_map[reoriented].append(row['Protein'])

duplicates = {motif: prots for motif, prots in reoriented_map.items() if len(prots) > 1}

print()
print('Motifs duplicated after reorientation (motif: [protein names]):')
print(f"Total duplicated motifs after reorientation: {len(duplicates)}")
for motif, prots in duplicates.items():
    print(f"{motif}: {prots}")
print()

for column in INVESTIGATION_COLUMNS:
    unique_techniques = []
    for record in motif_db[column].dropna().astype(str):
        for technique in record.split(';'):
            if technique not in unique_techniques:
                unique_techniques.append(technique)

    print(f'Techniques used in {column}:')
    for technique in unique_techniques:
        count = len(motif_db[motif_db[column].str.contains(technique, regex=False, na=False)])
        print(f"{technique}: {count}")
    print('Techniques used in combination:', len(motif_db[motif_db[column].str.contains(';', regex=False, na=False)]))
    print()

print('Biological processes concluded by the authors:')
unique_process = []
for process in motif_db['Biological process']:
    if process not in unique_process:
        unique_process.append(process)
for process in unique_process:
    print(process + ':', len(motif_db[motif_db['Biological process'] == process]))



In [ ]:
#@title Select high-quality motifs

def log_selected_criteria(criteria_dict, investigation_columns, record_count):
    """Print the filtering criteria and the number of selected records."""
    print("Selected criteria:")
    for key, value in criteria_dict.items():
        print(f"{key}: {value}")
    print(f"At least 1 investigation performed from {investigation_columns}")
    print()
    print(f"Selected records: {record_count}")


def filter_high_quality_motifs(motif_db,
                               criteria_dict,
                               investigation_columns,
                               verbose=True):
    """Select high-quality motifs using the current CMAscan dataset codes."""
    filtered = motif_db.copy()

    for key, values in criteria_dict.items():
        filtered = filtered[filtered[key].isin(values)]

    filtered = filtered[(filtered[investigation_columns] != NOT_INVESTIGATED).any(axis=1)]

    if verbose:
        log_selected_criteria(criteria_dict, investigation_columns, len(filtered))

    reoriented_motifs = [reorient_motif(motif) for motif in filtered['Motif']]

    if verbose:
        print('Selected motifs:', len(reoriented_motifs))

    return filtered, list(reoriented_motifs)


criteria_dict = {
    'Type': [TYPE_CANONICAL],
    'MDM': [MDM_CONFIRMED],
    'Biological process': ['CMA']
}

motif_db_hq, motifs_positive = filter_high_quality_motifs(
    motif_db,
    criteria_dict=criteria_dict,
    investigation_columns=INVESTIGATION_COLUMNS
)

print()

criteria_dict = {
    'Type': [TYPE_CANONICAL, TYPE_ATYPICAL],
    'MDM': [MDM_CONFIRMED],
    'Biological process': ['CMA']
}

motif_db_hq_ext, motifs_positive_ext = filter_high_quality_motifs(
    motif_db,
    criteria_dict=criteria_dict,
    investigation_columns=INVESTIGATION_COLUMNS
)



In [ ]:
#@title Trim motifs to 5 aa

def trim_motif(motif):
    """Trim a reoriented motif to 5 amino acids."""

    if len(motif) > 5:
        print(motif, 'trimmed to', motif[:5])

    return motif[:5]

motifs_positive = [trim_motif(motif) for motif in motifs_positive]
motifs_positive_ext = [trim_motif(motif) for motif in motifs_positive_ext]

print('Positive canonical motifs:', len(motifs_positive))
print('Positive canonical and atypical motifs:', len(motifs_positive_ext))


In [ ]:
#@title Unique residue combinations in the P dataset
# Assumes: motif_db is a pandas DataFrame with columns:
#   - "Motif reoriented"
#   - "Dataset"

import pandas as pd

MOTIF_COL = "Motif reoriented"
DATASET_COL = "Dataset"
POSITIVE_LABEL = DATASET_POSITIVE

NEG = set("DE")
POS = set("KR")
HYD = set("FLIV")


def is_canonical_composition(m: str, enforce_qn_first: bool = False) -> bool:
    if not isinstance(m, str):
        return False
    m = m.strip().upper()
    if len(m) != 5:
        return False
    if enforce_qn_first and m[0] not in {"Q", "N"}:
        return False

    neg_n = sum(aa in NEG for aa in m)
    pos_n = sum(aa in POS for aa in m)
    hyd_n = sum(aa in HYD for aa in m)

    if neg_n != 1:
        return False
    if not (1 <= pos_n <= 2):
        return False
    if not (1 <= hyd_n <= 2):
        return False

    return True


def motif_combination(m: str) -> str:
    m = m.strip().upper()
    return "".join(sorted(m))


# 1) Filter P dataset and clean motifs
pos_df = motif_db.loc[motif_db[DATASET_COL].eq(POSITIVE_LABEL), [MOTIF_COL, DATASET_COL]].copy()
pos_df[MOTIF_COL] = pos_df[MOTIF_COL].astype(str).str.strip().str.upper()
pos_df = pos_df.loc[pos_df[MOTIF_COL].ne("") & pos_df[MOTIF_COL].ne("NAN")]

# 2) Add canonical flag + combination
pos_df["is_canonical"] = pos_df[MOTIF_COL].apply(lambda x: is_canonical_composition(x, enforce_qn_first=False))
pos_df["combination_sorted"] = pos_df[MOTIF_COL].apply(motif_combination)

# 3) Compute uniqueness: sequences vs combinations
n_total = len(pos_df)
n_unique_sequences = pos_df[MOTIF_COL].nunique(dropna=True)
n_unique_combinations = pos_df["combination_sorted"].nunique(dropna=True)

print("P dataset summary (positive canonical set)")
print(f"Total motif records: {n_total}")
print(f"Unique motif sequences (order-specific): {n_unique_sequences}")
print(f"Unique combinations (order-independent multisets): {n_unique_combinations}")

if n_unique_sequences == n_unique_combinations:
    print("\nAnswer: YES — the unique sequences are also unique combinations (no AA-multiset duplicates).")
else:
    print("\nAnswer: NO — some different sequences share the same combination (same AA multiset, different order).")

# 4) Show collisions: combinations mapping to >1 distinct sequence
collision_tbl = (
    pos_df.groupby("combination_sorted")[MOTIF_COL]
          .nunique()
          .reset_index(name="n_distinct_sequences")
          .sort_values("n_distinct_sequences", ascending=False)
)
collision_tbl = collision_tbl.loc[collision_tbl["n_distinct_sequences"] > 1].copy()

if len(collision_tbl) == 0:
    print("\nNo combination collisions detected.")
else:
    print("\nCombination collisions (same combination, multiple sequences):")
    collision_tbl = collision_tbl.merge(
        pos_df.groupby("combination_sorted")[MOTIF_COL]
              .apply(lambda s: sorted(set(s)))
              .reset_index(name="sequences"),
        on="combination_sorted",
        how="left",
    )
    display(collision_tbl)
    print(collision_tbl['sequences'])

# 5) Canonical compliance report within the P dataset
canonical_counts = (
    pos_df["is_canonical"]
    .value_counts(dropna=False)
    .rename_axis("is_canonical")
    .reset_index(name="count")
)
print("\nCanonical rule compliance within the P dataset (composition-only rules):")
display(canonical_counts)

noncanonical = (
    pos_df.loc[~pos_df["is_canonical"], [MOTIF_COL, "combination_sorted"]]
          .drop_duplicates()
          .sort_values(MOTIF_COL)
)
if len(noncanonical) > 0:
    print("\nNon-canonical motifs found in the P dataset (per the composition rules above):")
    display(noncanonical)



In [ ]:
#@title Compare UniProt IDs with the motif structures file

import pandas as pd

def compare_uniprot_ids_between_sources(motif_db_hq, github_csv_url):
    """Compare UniProt IDs between motif_db_hq and the structures file."""
    # Load the GitHub CSV
    try:
        structures_df = pd.read_csv(github_csv_url)
        print(f"{len(set(structures_df['Uniprot ID'].tolist()))} UniProt IDs in motif_structures file")
    except Exception as e:
        print(f"Failed to load CSV from GitHub: {e}")
        return pd.DataFrame()

    # Clean motif_db_hq IDs (take first ID before semicolon)
    hq_ids = (
        motif_db_hq["UniProt ID"]
        .dropna()
        .astype(str)
        .str.strip()
        .str.split(";")
        .str[0]
        .str.strip()
        .unique()
    )

    # Clean GitHub IDs (also take first ID before semicolon)
    struct_ids = (
        structures_df["Uniprot ID"]
        .dropna()
        .astype(str)
        .str.strip()
        .str.split(";")
        .str[0]
        .str.strip()
        .unique()
    )

    hq_set = set(hq_ids)
    struct_set = set(struct_ids)

    only_in_hq = sorted(hq_set - struct_set)
    only_in_struct = sorted(struct_set - hq_set)

    # Build output table
    missing_df = pd.DataFrame({
        "Missing in": ["motif_structures_v0.6.csv"] * len(only_in_hq) +
                      ["motif_db_hq"] * len(only_in_struct),
        "UniProt ID": only_in_hq + only_in_struct
    })

    if missing_df.empty:
        print("All UniProt IDs match between motif_db_hq and motif_structures_v0.6.csv.")
    else:
        print("Differences were found between the two sources:")
        display(missing_df)

    return missing_df

github_url = resolve_repo_file('dataset/motif_structures_v0.6.csv')
missing_uniprot_ids = compare_uniprot_ids_between_sources(motif_db_hq, github_url)


In [ ]:
#@title Positional amino acid counts

import pandas as pd
from collections import defaultdict

def count_positional_preferences_df(motifs):
    """
    Creates a DataFrame with amino acid counts at each motif position.

    Parameters:
    - motifs: List of motif sequences (each a string of length 5)

    Returns:
    - DataFrame: Amino acids as rows, positions (1,2,3,4,5) as columns, values as counts.
    """

    # Initialize dictionary to store counts
    position_counts = {pos: defaultdict(int) for pos in range(1, 6)}  # Positions 1 to 5

    # Count occurrences at each position
    for motif in motifs:
        for pos, aa in enumerate(motif, start=1):  # Position 1 to 5
            position_counts[pos][aa] += 1

    # Convert to DataFrame
    df = pd.DataFrame(position_counts).fillna(0).astype(int)  # Fill missing values with 0

    return df

reoriented_motifs = [reorient_motif(motif) for motif in motif_db_hq['Motif']]

aa2plot_df = count_positional_preferences_df(reoriented_motifs)

# Define desired amino acid order
aa_order = ['Q', 'N', 'R', 'K', 'E', 'D', 'F', 'L', 'I', 'V']

# Reorder rows in the DataFrame
aa2plot_df = aa2plot_df.reindex(aa_order).fillna(0).astype(int)

# Export and display
aa2plot_df.to_csv('positional_counts.csv', index=True)
aa2plot_df


In [ ]:
#@title Positional amino acid counts - canonical + atypical

# Reorient motifs if needed
reoriented_motifs_ext = [reorient_motif(motif) for motif in motifs_positive_ext]

# Compute positional counts
aa2plot_ext_df = count_positional_preferences_df(reoriented_motifs_ext)

# Desired full amino-acid order
aa_order_full = ['A', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L',
                 'M', 'N', 'P', 'Q', 'R', 'S', 'T', 'V', 'W', 'Y']

# Reorder and export
aa2plot_ext_df = aa2plot_ext_df.reindex(aa_order_full).fillna(0).astype(int)
aa2plot_ext_df.to_csv('positional_counts_ext.csv', index=True)

aa2plot_ext_df


In [ ]:
#@title Publication figure settings

mpl.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Liberation Sans', 'DejaVu Sans'],
    'font.size': 7,
    'axes.linewidth': 0.75,
    'xtick.major.width': 0.75,
    'ytick.major.width': 0.75,
    'lines.linewidth': 1.0,
    'pdf.fonttype': 42,
    'svg.fonttype': 'none',
})



In [ ]:
#@title Positional amino acid preferences barplots

def plot_positional_preferences_with_other_legend(
    df_top,
    df_bottom,
    width_mode='full_width',
    fig_height=None,
    bar_height=0.6,
    wspace_value=0.1,
    hspace_value=0.7,
    dpi=300,
    title_fontsize=7,
    tick_fontsize=7,
    label_fontsize=7,
    rowlabel_fontsize=7,
    legend_fontsize=7,
    legend_handleheight=0.4,
    save_path='positional_preferences_combined_other_legend.svg',
    save_format='svg',
    include_other=False,
    x_max_amide=50,
    x_max_other=25,
    top_row_label='Canonical',
    bottom_row_label='Canonical + atypical',
    panel_top='A',
    panel_bottom='B',
    panel_x=-0.28,
    panel_y_top=1.4,
    panel_y_bottom=1.1,
    transparent=True,
):
    import matplotlib.patches as mpatches
    from collections import OrderedDict

    fig_width, fig_height = get_publication_figsize(
        default_width=8.5,
        default_height=3.0,
        width_mode=width_mode,
        fig_height=fig_height,
    )
    save_path = normalise_save_path(save_path, save_format)

    aa_color = {
        'Q': '#9C755F',
        'N': '#7E57C2',
        'K': '#4E79A7',
        'R': '#F28E2B',
        'D': '#76B7B2',
        'E': '#E15759',
        'F': '#59A14F',
        'L': '#EDC948',
        'I': '#B07AA1',
        'V': '#FF9DA7',
    }

    categories = OrderedDict(
        {
            'Amide': ['Q', 'N'],
            'Positive': ['K', 'R'],
            'Negative': ['D', 'E'],
            'Hydrophobic': ['F', 'L', 'I', 'V'],
        }
    )

    df_top = df_top.copy()
    df_bottom = df_bottom.copy()
    df_top.columns = df_top.columns.astype(str)
    df_bottom.columns = df_bottom.columns.astype(str)

    required_cols = ['1', '2', '3', '4', '5']
    missing_top = [c for c in required_cols if c not in df_top.columns]
    missing_bottom = [c for c in required_cols if c not in df_bottom.columns]
    if missing_top or missing_bottom:
        raise ValueError(
            f'Missing required motif-position columns: top={missing_top}, bottom={missing_bottom}'
        )

    if include_other:
        predefined = {aa for grp in categories.values() for aa in grp}
        union_index = pd.Index(df_top.index).union(pd.Index(df_bottom.index))
        other = []
        for aa in union_index:
            if aa in predefined:
                continue
            top_sum = df_top.loc[aa].sum() if aa in df_top.index else 0
            bottom_sum = df_bottom.loc[aa].sum() if aa in df_bottom.index else 0
            if (top_sum + bottom_sum) > 0:
                other.append(aa)
        categories['Other'] = other

    other_legend_color = '#BDBDBD'
    other_color_map = {}
    if 'Other' in categories:
        other_color_map = {aa: other_legend_color for aa in categories['Other']}

    def _prep_df(df):
        return df.loc[:, required_cols][['5', '4', '3', '2', '1']]

    df_top_p = _prep_df(df_top)
    df_bottom_p = _prep_df(df_bottom)

    y_pos = np.arange(len(df_top_p.columns))
    ncols = len(categories)

    fig, axes = plt.subplots(
        nrows=2,
        ncols=ncols,
        figsize=(fig_width, fig_height),
        sharey=True,
    )
    plt.subplots_adjust(wspace=wspace_value, hspace=hspace_value, top=0.86)

    def _plot_row(df_row, ax_row, show_legends, show_titles):
        for ax, (cat, aas) in zip(ax_row, categories.items()):
            left = np.zeros(len(y_pos))
            handles, labels = [], []

            for aa in aas:
                if aa not in df_row.index:
                    continue
                vals = df_row.loc[aa].values
                colour = aa_color.get(aa, other_color_map.get(aa, '#BDBDBD'))
                bar = ax.barh(
                    y=y_pos,
                    width=vals,
                    height=bar_height,
                    left=left,
                    color=colour,
                    edgecolor='black',
                    linewidth=0.3,
                    label=aa,
                )
                left += vals
                handles.append(bar[0])
                labels.append(aa)

            ax.set_title(cat if show_titles else '', fontsize=title_fontsize)
            ax.set_yticks(y_pos)
            ax.set_yticklabels(df_row.columns)

            if cat == 'Amide':
                ax.set_xlim(0, x_max_amide)
                ax.set_xticks(np.arange(0, x_max_amide + 1, 10))
            else:
                ax.set_xlim(0, x_max_other)
                ax.set_xticks(np.arange(0, x_max_other + 1, 5))

            ax.tick_params(axis='both', labelsize=tick_fontsize)
            ax.grid(axis='x', linestyle='--', linewidth=0.4, alpha=0.5)
            ax.spines[['top', 'right', 'left']].set_visible(False)

            if show_legends and labels:
                if cat == 'Other':
                    other_label = f"Other ({', '.join(labels)})"
                    other_handle = mpatches.Patch(
                        facecolor=other_legend_color,
                        edgecolor='black',
                        linewidth=0.3,
                        label=other_label,
                    )
                    ax.legend(
                        handles=[other_handle],
                        labels=[other_label],
                        loc='upper center',
                        bbox_to_anchor=(0.5, -0.35),
                        fontsize=legend_fontsize,
                        frameon=False,
                        ncol=1,
                        handleheight=legend_handleheight,
                        handlelength=0.8,
                        handletextpad=0.3,
                        columnspacing=0.5,
                    )
                else:
                    ax.legend(
                        handles=handles,
                        labels=labels,
                        loc='upper center',
                        bbox_to_anchor=(0.5, -0.35),
                        fontsize=legend_fontsize,
                        frameon=False,
                        ncol=len(labels),
                        handleheight=legend_handleheight,
                        handlelength=0.8,
                        handletextpad=0.3,
                        columnspacing=0.5,
                    )

    _plot_row(df_top_p, axes[0], show_legends=False, show_titles=True)
    _plot_row(df_bottom_p, axes[1], show_legends=True, show_titles=False)

    axes[0, 0].set_ylabel('Motif Position', fontsize=label_fontsize)
    axes[1, 0].set_ylabel('Motif Position', fontsize=label_fontsize)

    top_boxes = [ax.get_position() for ax in axes[0]]
    bottom_boxes = [ax.get_position() for ax in axes[1]]
    top_x_center = (top_boxes[0].x0 + top_boxes[-1].x1) / 2
    bottom_x_center = (bottom_boxes[0].x0 + bottom_boxes[-1].x1) / 2

    top_y = fig.transFigure.inverted().transform(
        axes[0, 0].transAxes.transform((0, panel_y_top))
    )[1]
    bottom_y = fig.transFigure.inverted().transform(
        axes[1, 0].transAxes.transform((0, panel_y_bottom))
    )[1]

    fig.text(
        top_x_center,
        top_y,
        top_row_label,
        ha='center',
        va='bottom',
        fontsize=rowlabel_fontsize,
        fontweight='bold',
    )
    fig.text(
        bottom_x_center,
        bottom_y,
        bottom_row_label,
        ha='center',
        va='bottom',
        fontsize=rowlabel_fontsize,
        fontweight='bold',
    )

    axes[0, 0].text(
        panel_x,
        panel_y_top,
        panel_top,
        transform=axes[0, 0].transAxes,
        fontsize=rowlabel_fontsize + 1,
        fontweight='bold',
        ha='left',
        va='bottom',
    )
    axes[1, 0].text(
        panel_x,
        panel_y_bottom,
        panel_bottom,
        transform=axes[1, 0].transAxes,
        fontsize=rowlabel_fontsize + 1,
        fontweight='bold',
        ha='left',
        va='bottom',
    )

    fig.savefig(
        save_path,
        format=save_format,
        dpi=dpi,
        bbox_inches='tight',
        transparent=transparent,
    )

    plt.show()
    print(f'Figure saved as {save_path} (format={save_format})')


#=== EXECUTION =================================================================
plot_positional_preferences_with_other_legend(
    aa2plot_df,
    aa2plot_ext_df,
    width_mode='full_width',
    fig_height=3.8,
    save_path='positional_preferences_combined_other_legend.svg',
    include_other=True,
    x_max_amide=50,
    x_max_other=25,
    panel_y_top=1.3,
    panel_y_bottom=1.1,
)



In [ ]:
#@title Positive atypical motifs (PA)

motif_db[motif_db['Dataset'] == DATASET_POSITIVE_ATYPICAL]



In [ ]:
#@title Residues outside the canonical set in PA motifs

new_aas = {}
allowed = set(['Q', 'N', 'E', 'D', 'R', 'K', 'F', 'L', 'I', 'V'])

for motif in motif_db.loc[motif_db["Dataset"] == DATASET_POSITIVE_ATYPICAL, "Motif reoriented"].dropna().astype(str):
    motif = motif.strip().upper()
    for aa in motif:
        if aa not in allowed:
            new_aas[aa] = new_aas.get(aa, 0) + 1

print(new_aas)



In [ ]:
#@title Atypical motifs without atypical amino acids

# Count and print motifs from the PA dataset that contain no atypical amino acids,
# but still fail the canonical composition rules.

NEG = set("DE")
POS = set("KR")
HYD = set("FLIV")
allowed = set(['Q', 'N', 'E', 'D', 'R', 'K', 'F', 'L', 'I', 'V'])


def is_canonical_composition(m: str) -> bool:
    m = m.strip().upper()
    if len(m) != 5:
        return False
    neg_n = sum(aa in NEG for aa in m)
    pos_n = sum(aa in POS for aa in m)
    hyd_n = sum(aa in HYD for aa in m)
    return (neg_n == 1) and (1 <= pos_n <= 2) and (1 <= hyd_n <= 2)


def has_no_atypical_aas(m: str) -> bool:
    return all(aa in allowed for aa in m)

motifs = (
    motif_db.loc[motif_db["Dataset"] == DATASET_POSITIVE_ATYPICAL, "Motif reoriented"]
    .dropna()
    .astype(str)
    .str.strip()
    .str.upper()
)

motifs = motifs[motifs.str.len() == 5]

no_atypical_but_noncanonical = [
    m for m in motifs
    if has_no_atypical_aas(m) and (not is_canonical_composition(m))
]

print(f"Count: {len(no_atypical_but_noncanonical)}")
print("Motifs:")
for m in sorted(set(no_atypical_but_noncanonical)):
    print(m)



In [ ]:
#@title Background amino acid composition

from Bio import SeqIO
import requests
from io import StringIO
from collections import defaultdict

def calculate_aa_distribution_from_source(fasta_source):
    """Calculate amino acid frequencies from a local FASTA file or URL."""
    print(f'Loading: {fasta_source}')

    if str(fasta_source).startswith('http'):
        response = requests.get(fasta_source)
        response.raise_for_status()
        handle = StringIO(response.text)
    else:
        handle = open(fasta_source, 'r')

    total_aa = 0
    aa_counts = defaultdict(int)
    n = 0

    try:
        for record in SeqIO.parse(handle, 'fasta'):
            n += 1
            seq = str(record.seq)
            for aa in seq:
                aa_counts[aa] += 1
                total_aa += 1
    finally:
        handle.close()

    print(f'Number of proteins: {n}')
    aa_distribution = {aa: count / total_aa for aa, count in aa_counts.items()}
    return aa_distribution

mammalian_mean_aa_distribution = {
    'S': 0.0833640853959927,
    'T': 0.0534707922749892,
    'D': 0.0473298742499409,
    'Q': 0.0469573385466771,
    'Y': 0.0268749552304211,
    'F': 0.0373789229889875,
    'G': 0.0654433309782482,
    'P': 0.0623692840533557,
    'R': 0.0566692634635304,
    'L': 0.100605377772131,
    'V': 0.0605991043212125,
    'M': 0.0216626159993309,
    'C': 0.0231585862417482,
    'H': 0.0260597646198473,
    'E': 0.0694340464165267,
    'N': 0.0356027235615439,
    'A': 0.0699340171989205,
    'K': 0.0567922470525259,
    'W': 0.012267877287158,
    'I': 0.0440021611837513,
    'U': 2.5774934115491795e-06,
    'X': 2.07837023259903e-05,
    'Z': 7.978272463796199e-07,
    'B': 7.669243924527316e-07
}

human_proteome_fasta_url = (
    'https://rest.uniprot.org/uniprotkb/stream?compressed=false&format=fasta&query=%28proteome%3AUP000005640%29%20AND%20%28reviewed%3Atrue%29'
)
human_aa_distribution = calculate_aa_distribution_from_source(human_proteome_fasta_url)

human_aa_distribution


In [ ]:
#@title Build PSSM matrices
from Bio import motifs

# Define the alphabet and amino acid distribution
alphabet = 'RHKDESTNQCGPAVILMFYW'

# Laplace parameter k
k = 0.0001

aa_distribution = human_aa_distribution

# Calculate the PSSM matrices
pssm_matrix = calculate_pssm_matrix(motifs_positive, alphabet, aa_distribution, k)
pssm_matrix_expanded = calculate_pssm_matrix(motifs_positive_ext, alphabet, aa_distribution, k)
print('PSSM:', len(motifs_positive))
print('PSSM extended:', len(motifs_positive_ext))

# Calculate the charge-based PSSM matrices
pssm_charge_matrix = calculate_pssm_charge_matrix(motifs_positive, aa_distribution, k)
pssm_charge_matrix_expanded = calculate_pssm_charge_matrix(motifs_positive_ext, aa_distribution, k)


import pickle

# Save PSSM matrix object to file
with open('pssm_matrix.pkl', 'wb') as f:
    pickle.dump(pssm_matrix, f)

with open('pssm_expanded.pkl', 'wb') as f:
    pickle.dump(pssm_matrix_expanded, f)

with open('pssm_charge.pkl', 'wb') as f:
    pickle.dump(pssm_charge_matrix, f)


In [ ]:
terlecky_set = ['KFERQ', 'KFEPQ', 'KFERP', 'KFPRQ', 'KFERR', 'PFERQ', 'KPERQ']

for test in terlecky_set:
    print(test)
    print(calculate_pssm_score(test, pssm_matrix))
    print(calculate_pssm_score(test, pssm_matrix_expanded))
    print(calculate_pssm_score(charge_transformation(test), pssm_charge_matrix))
    print(calculate_pssm_score(charge_transformation(test), pssm_charge_matrix_expanded))
    print()


In [ ]:
#@title Lowest score in each PSSM
# Assumes:
# - motifs_positive : list[str]
# - pssm, pssm_ext, pssm_charge : pandas.DataFrame or compatible score matrices
# - calculate_pssm_score(motif, pssm_matrix) : callable
# - charge_transformation(motif) : callable

pssms = {
    "pssm": pssm_matrix,
    "pssm_ext": pssm_matrix_expanded,
    "pssm_charge": pssm_charge_matrix,
}

# Initialize with +inf so any real score will be smaller
lowest_scores = {name: float("inf") for name in pssms}

for motif in motifs_positive:
    for name, matrix in pssms.items():
        # Apply charge transformation only for the charge-specific PSSM
        motif_input = charge_transformation(motif) if name == "pssm_charge" else motif
        score = calculate_pssm_score(motif_input, matrix)

        if score < lowest_scores[name]:
            lowest_scores[name] = score

# Print results
for name, score in lowest_scores.items():
    print(f"{name}: lowest = {score:.2f}")


In [ ]:
#@title Generate all canonical motifs

import pandas as pd
import itertools

def generate_canonical_permutations(amino_acids, pssm_matrix, pssm_matrix_expanded, motif_length=5, positive_motifs=None):
    """
    Generate all possible canonical motifs, validate them, and return a DataFrame.

    Args:
        amino_acids (list): List of amino acids to create motifs.
        pssm_matrix (dict): PSSM matrix to calculate scores.
        pssm_matrix_expanded (dict): Extended PSSM matrix.
        motif_length (int): Length of the motif (default: 5).
        positive_motifs (list): List of experimentally validated motifs (default: None).

    Returns:
        pd.DataFrame: DataFrame with motif, validation status, and PSSM scores.
    """
    # Generate all motif combinations
    all_combinations = itertools.product(amino_acids, repeat=motif_length)
    canonical_motifs = [''.join(comb) for comb in all_combinations if is_canonical(''.join(comb))]
    canonical_motifs = [reorient_motif(motif) for motif in canonical_motifs]
    canonical_motifs = list(set(canonical_motifs))

    # Compute motif scores
    motif_scores = calculate_motif_scores(canonical_motifs, pssm_matrix)
    motif_scores_ext = calculate_motif_scores(canonical_motifs, pssm_matrix_expanded)

    # Prepare validation check
    positive_motifs = set(positive_motifs) if positive_motifs else set()

    # Create DataFrame
    df = pd.DataFrame({
        "motif": canonical_motifs,
        "validated": [1 if motif in positive_motifs else 0 for motif in canonical_motifs],
        "PSSM score": [motif_scores.get(motif, None) for motif in canonical_motifs],
        "PSSM score ext": [motif_scores_ext.get(motif, None) for motif in canonical_motifs]
    })

    return df

def calculate_motif_scores(motifs, pssm_matrix):
    """
    Calculate PSSM scores for the given motifs.

    Args:
        motifs (list): List of canonical motifs.
        pssm_matrix (dict): PSSM matrix for scoring.

    Returns:
        dict: Mapping from motif to PSSM score.
    """
    motif_scores = {}
    for motif in motifs:
        reoriented_motif = reorient_motif(motif)
        if reoriented_motif not in motif_scores:
            motif_scores[reoriented_motif] = calculate_pssm_score(reoriented_motif, pssm_matrix)
    return motif_scores

# Define amino acids
amino_acids = ["Q", "N", "K", "R", "D", "E", "F", "L", "I", "V"]

# Define positive motifs (e.g., from experimental validation)
positive_motifs = [reorient_motif(motif) for motif in motif_db_hq['Motif']]

# Generate canonical motifs and store in DataFrame
canonical_permutations_df = generate_canonical_permutations(amino_acids, pssm_matrix, pssm_matrix_expanded, positive_motifs=positive_motifs)
print("Number of possible permutations:", len(canonical_permutations_df))


In [ ]:
#@title Build a charge-based motif set

import pandas as pd

def generate_charge_permutations_table(motifs_positive, motifs):
    """
    Calculate the presence of each possible charge permutation in the list of charge motifs
    and return a sorted DataFrame.

    Args:
        motifs_positive (list): A list of motifs for positive sequences.
        motifs (list): A list of motifs for generating possible charge permutations.

    Returns:
        pd.DataFrame: A DataFrame with possible charge permutations and their presence, sorted by presence.
    """
    # Create lists and sets of charge motifs and possible charge permutations
    charge_motifs = [charge_transformation(motif) for motif in motifs_positive]
    possible_charge_permutations = set([charge_transformation(reorient_motif(motif)) for motif in motifs])

    # Prepare data for the DataFrame
    data = []
    for motif in possible_charge_permutations:
        presence = sum(1 for charge_motif in charge_motifs if charge_motif == motif)
        data.append({
            'possible_charge_permutation': motif,
            'presence': presence,
            'PSSM charge score': round(calculate_pssm_score(motif, pssm_charge_matrix), 2)
        })

    # Create a DataFrame
    df = pd.DataFrame(data)

    return df

charge_motif_table = generate_charge_permutations_table(motifs_positive, canonical_permutations_df['motif']).sort_values(by='presence', ascending=False)
charge_motif_table.to_csv('Table 3 - charge_permutations.csv', index=False)
charge_motif_table


In [ ]:
#@title Plot helper

def plot_motif_scores(
    df,
    target_col,
    score_col,
    x_label='Motifs',
    width_mode='full_width',
    fig_height=None,
    dot_size=10,
    jitter=0.04,
    save_path=None,
    save_format='svg',
    dpi=300,
    transparent=True,
):
    """Generate a dot plot of motif scores with publication-ready figure settings."""

    fig_width, fig_height = get_publication_figsize(
        default_width=3.0,
        default_height=5.0,
        width_mode=width_mode,
        fig_height=fig_height,
    )
    save_path = normalise_save_path(save_path, save_format)

    scores = df[score_col]
    targets = df[target_col]
    x_positions = np.random.normal(1, jitter, size=len(scores))

    neg_mask = targets == 0
    pos_mask = targets != 0

    fig, ax = plt.subplots(figsize=(fig_width, fig_height))
    ax.scatter(x_positions[neg_mask], scores[neg_mask], c='white', edgecolors='black', alpha=0.4, s=dot_size)
    ax.scatter(x_positions[pos_mask], scores[pos_mask], c='blue', edgecolors='black', alpha=1, s=dot_size)

    ax.set_xticks([])
    ax.set_xticklabels([])
    ax.set_ylabel(score_col)
    ax.grid(False)

    fig.tight_layout()

    if save_path:
        fig.savefig(
            save_path,
            format=save_format,
            dpi=dpi,
            bbox_inches='tight',
            transparent=transparent,
        )
        print(f'Figure saved as {save_path}.')

    plt.show()



In [ ]:
#@title Violin plot of cPSSM and ePSSM

def plot_violin_pssm_vs_ext(
    df,
    col1='PSSM score',
    col2='PSSM score ext',
    width_mode='full_width',
    fig_height=None,
    font_scale=0.8,
    dpi=300,
    save_path=None,
    save_format='svg',
    transparent=True,
):
    """Violin plot of cPSSM and ePSSM scores with publication-ready export settings."""

    fig_width, fig_height = get_publication_figsize(
        default_width=3.0,
        default_height=3.0,
        width_mode=width_mode,
        fig_height=fig_height,
    )
    save_path = normalise_save_path(save_path, save_format)

    melted_df = df[[col1, col2]].copy()
    melted_df.columns = ['cPSSM', 'ePSSM']
    melted_df = melted_df.melt(var_name='Score Type', value_name='Score')

    sns.set(style='whitegrid')
    sns.set_context('notebook', font_scale=font_scale)

    fig, ax = plt.subplots(figsize=(fig_width, fig_height))
    sns.violinplot(
        x='Score Type',
        y='Score',
        data=melted_df,
        inner=None,
        linewidth=0.8,
        ax=ax,
    )

    ax.set_ylabel('Score')
    ax.set_xlabel('')
    ax.grid(True)
    fig.tight_layout()

    if save_path:
        fig.savefig(
            save_path,
            format=save_format,
            dpi=dpi,
            bbox_inches='tight',
            transparent=transparent,
        )
        print(f'Figure saved as {save_path}.')

    plt.show()


plot_violin_pssm_vs_ext(
    canonical_permutations_df,
    col1='PSSM score',
    col2='PSSM score ext',
    width_mode='two_column',
    fig_height=3.0,
    font_scale=0.8,
    dpi=300,
    save_path='PSSM_spectrum_comparison.svg',
)

for col, label in [('PSSM score', 'cPSSM'), ('PSSM score ext', 'ePSSM')]:
    col_min = canonical_permutations_df[col].min()
    col_max = canonical_permutations_df[col].max()
    col_median = canonical_permutations_df[col].median()
    print(f'{label}: min = {col_min:.2f}, max = {col_max:.2f}, median = {col_median:.2f}')



In [ ]:
#@title LOOCV helper functions

def build_pssm_matrix(train_set, alphabet, background_distribution, k=0.0001, mode='aa'):
    """
    Build a PSSM matrix based on the specified mode.

    Args:
        train_set (list): List of motifs to train on.
        alphabet (str): Amino acid alphabet.
        background_distribution (dict): Background amino acid distribution.
        k (float): Pseudocount smoothing factor.
        mode (str): 'aa' for amino acid, 'charge' for charge-transformed matrix.

    Returns:
        dict: PSSM matrix.
    """
    if mode == 'aa':
        return calculate_pssm_matrix(train_set, alphabet, background_distribution, k)
    elif mode == 'charge':
        return calculate_pssm_charge_matrix(train_set, background_distribution, k)
    else:
        raise ValueError("Mode must be 'aa' or 'charge'")


def score_motifs(test_set, pssm_matrix, mode='aa'):
    """
    Score each motif in the test set against the provided PSSM matrix.

    Args:
        test_set (list): Motifs to score.
        pssm_matrix (dict): Precomputed PSSM matrix.
        mode (str): 'aa' or 'charge'.

    Returns:
        dict: {motif: score}
    """
    scores = {}
    for motif in test_set:
        query = charge_transformation(motif) if mode == 'charge' else motif
        scores[motif] = calculate_pssm_score(query, pssm_matrix)
    return scores


def run_custom_loocv(train_set, test_set, alphabet, background_distribution, k=0.0001, mode='aa'):
    """
    Train on a full training set, test on an external or partial test set.

    Used for Goal 2: train on canonicals+atypicals, test on canonicals.

    Returns:
        dict: {motif: score}
    """
    pssm_matrix = build_pssm_matrix(train_set, alphabet, background_distribution, k, mode)
    return score_motifs(test_set, pssm_matrix, mode)


def run_partial_loocv(train, test_subset, alphabet, background_distribution, k=0.0001, mode='aa'):
    """
    True LOOCV. For each motif in test_subset, build matrix without that ONE occurrence.

    Returns:
        dict: {motif: score}
    """
    results = {}

    for test_motif in test_subset:
        reduced_train = list(train)
        reduced_train.remove(test_motif)

        scores = run_custom_loocv(
            reduced_train,
            [test_motif],
            alphabet,
            background_distribution,
            k,
            mode
        )
        results.update(scores)

    return results



In [ ]:
#@title LOOCV settings

alphabet = 'QWERTYIPASDFGHKLCVNM'
background = human_aa_distribution
k = 0.0001


In [ ]:
#@title LOOCV - canonical only

aa_scores = run_partial_loocv(
    train=motifs_positive,
    test_subset=motifs_positive,
    alphabet=alphabet,
    background_distribution=background,
    k=k,
    mode='aa'
)
print('Train:', len(motifs_positive))
print("Test:", len(motifs_positive))


In [ ]:
#@title LOOCV - canonical + atypical in training, canonical in test

aa_scores_ext = run_custom_loocv(
    train_set=motifs_positive_ext,
    test_set=motifs_positive,
    alphabet=alphabet,
    background_distribution=background,
    k=k,
    mode='aa'
)
print('Train:', len(motifs_positive_ext))
print("Test:", len(motifs_positive))


In [ ]:
#@title LOOCV - canonical + atypical in training, all motifs in test

aa_scores_ext_all = run_custom_loocv(
    train_set=motifs_positive_ext,
    test_set=motifs_positive_ext,
    alphabet=alphabet,
    background_distribution=background,
    k=k,
    mode='aa'
)
print('Train:', len(motifs_positive_ext))
print("Test:", len(motifs_positive_ext))


In [ ]:
#@title LOOCV - charge-based, canonical motifs

charge_scores = run_partial_loocv(
    train=motifs_positive,
    test_subset=motifs_positive,
    alphabet=alphabet,
    background_distribution=background,
    k=k,
    mode='charge'
)


In [ ]:
#@title Plot LOOCV

def plot_stacked_loocv(
    aa_scores,
    aa_scores_ext,
    threshold=0,
    width_mode='full_width',
    fig_height=None,
    hspace=0.15,
    save_path='loocv_plot.svg',
    save_format='svg',
    dpi=300,
    transparent=True,
):
    """Plot vertically stacked LOOCV results with motif order fixed by canonical-only scores."""

    fig_width, fig_height = get_publication_figsize(
        default_width=9.0,
        default_height=5.0,
        width_mode=width_mode,
        fig_height=fig_height,
    )
    save_path = normalise_save_path(save_path, save_format)

    sorted_motifs = sorted(aa_scores, key=aa_scores.get)
    y1 = [aa_scores[m] for m in sorted_motifs]
    y2 = [aa_scores_ext[m] for m in sorted_motifs]

    fig, axs = plt.subplots(2, 1, figsize=(fig_width, fig_height), sharex=True, gridspec_kw={'hspace': hspace})

    for ax, scores in zip(axs, [y1, y2]):
        colors = ['green' if s >= threshold else 'red' for s in scores]
        ax.scatter(range(len(scores)), scores, color=colors, s=18)
        ax.axhline(y=threshold, color='gray', linestyle='--', linewidth=1)
        ax.tick_params(axis='y', labelsize=7)

    axs[1].set_xticks(range(len(sorted_motifs)))
    axs[1].set_xticklabels(sorted_motifs, rotation=90, fontsize=7)

    plt.tight_layout()

    if save_path:
        fig.savefig(
            save_path,
            format=save_format,
            dpi=dpi,
            bbox_inches='tight',
            transparent=transparent,
        )
        print(f'Figure saved as {save_path}.')

    plt.show()

    def print_stats(name, values):
        values = list(values)
        total = len(values)
        passed = sum(v >= threshold for v in values)
        above_zero = [v for v in values if v > 0]

        print(f'\n{name}:')
        print(f'  TPR (≥ {threshold:.1f}): {passed}/{total} ({passed/total:.0%})')
        print(f'  Min: {min(values):.2f}')
        print(f'  Max: {max(values):.2f}')
        if above_zero:
            print(f'  Min > 0: {min(above_zero):.2f}')
            print(f'  Max > 0: {max(above_zero):.2f}')
        else:
            print('  No values > 0')
        print(f'  Median: {np.median(values)}')
        print(f'  Mean: {np.mean(values)}')

    print('LOOCV Summary')
    print_stats('cPSSM', aa_scores.values())
    print_stats('ePSSM', aa_scores_ext.values())


plot_stacked_loocv(
    aa_scores,
    aa_scores_ext,
    width_mode='full_width',
    fig_height=3.5,
    hspace=0.1,
    save_path='loocv_plot.svg',
)


In [ ]:
plot_stacked_loocv(
    aa_scores_ext_all,
    aa_scores_ext_all,
    width_mode='full_width',
    fig_height=3.5,
    hspace=0.1
)


In [ ]:
import pandas as pd

# aa_scores_ext_all, aa_scores_ext, aa_scores are dicts: {motif: score}

df_scores = (
    pd.DataFrame.from_dict(aa_scores_ext_all, orient="index", columns=["score_ext_all"])
    .join(pd.DataFrame.from_dict(aa_scores_ext, orient="index", columns=["score_ext"]), how="outer")
    .join(pd.DataFrame.from_dict(aa_scores, orient="index", columns=["score"]), how="outer")
    .rename_axis("motif")
    .reset_index()
)

# Optional: sort by one of the columns (keeps NaNs at the bottom by default)
# df_scores = df_scores.sort_values(by="score_ext_all", ascending=False)

df_scores = df_scores.round(2)
df_scores



In [ ]:
#@title Plot TPR

def plot_tpr_curves(
    aa_scores,
    aa_scores_ext,
    thresholds=None,
    width_mode='full_width',
    fig_height=None,
    fontsize=7,
    lw=1.0,
    dpi=300,
    legend_style='outside_right',
    save_path=None,
    save_format='svg',
    transparent=True,
):
    """Plot TPR as a function of the PSSM threshold."""

    fig_width, fig_height = get_publication_figsize(
        default_width=3.0,
        default_height=3.0,
        width_mode=width_mode,
        fig_height=fig_height,
    )
    save_path = normalise_save_path(save_path, save_format)

    if thresholds is None:
        thresholds = np.arange(-10, 16, 1)
    thresholds = np.asarray(thresholds)

    def compute_tpr(scores, thresh):
        values = np.asarray(list(scores.values()), dtype=float)
        return np.mean(values >= thresh) if values.size else np.nan

    tpr_cpssm = [compute_tpr(aa_scores, t) for t in thresholds]
    tpr_epssm = [compute_tpr(aa_scores_ext, t) for t in thresholds]

    fig, ax = plt.subplots(figsize=(fig_width, fig_height))
    l1, = ax.plot(thresholds, tpr_cpssm, linestyle='-', linewidth=lw, label='cPSSM')
    l2, = ax.plot(thresholds, tpr_epssm, linestyle='--', linewidth=lw, label='ePSSM')

    ax.set_xlabel('PSSM threshold', fontsize=fontsize)
    ax.set_ylabel('True Positive Rate', fontsize=fontsize)
    ax.set_xlim(-10, 15)
    ax.set_ylim(-0.05, 1.05)
    ax.tick_params(axis='both', which='both', labelsize=fontsize - 1, direction='out', length=3, width=0.8)
    for spine in ('top', 'right'):
        ax.spines[spine].set_visible(False)
    for spine in ('left', 'bottom'):
        ax.spines[spine].set_linewidth(1.0)
    ax.grid(True, linestyle='--', linewidth=0.6, alpha=0.4)

    if legend_style == 'outside_right':
        leg = ax.legend(fontsize=fontsize - 1, frameon=False, handlelength=2.5, loc='center left', bbox_to_anchor=(1.02, 0.5), borderaxespad=0.2)
    elif legend_style == 'below':
        leg = ax.legend(fontsize=fontsize - 1, frameon=False, handlelength=2.5, loc='upper center', bbox_to_anchor=(0.5, -0.12), ncol=2)
    elif legend_style == 'inside_tr':
        leg = ax.legend(fontsize=fontsize - 1, frameon=True, handlelength=2.5, loc='upper right')
        leg.get_frame().set_alpha(0.85)
        leg.get_frame().set_edgecolor('none')
        leg.get_frame().set_facecolor('white')
    elif legend_style == 'inside_bl':
        leg = ax.legend(fontsize=fontsize - 1, frameon=False, handlelength=2.5, loc='lower left')
    elif legend_style == 'inline':
        for line, txt in zip((l1, l2), ('cPSSM', 'ePSSM')):
            xdata = line.get_xdata()
            ydata = line.get_ydata()
            idx = np.where(np.isfinite(ydata))[0][-1]
            x, y = xdata[idx], ydata[idx]
            ax.annotate(
                txt,
                (x, y),
                xytext=(3, 0),
                textcoords='offset points',
                va='center',
                ha='left',
                fontsize=fontsize - 1,
                path_effects=[pe.withStroke(linewidth=3, foreground='white', alpha=0.8)],
            )
        leg = None
    else:
        leg = ax.legend(fontsize=fontsize - 1, frameon=False, handlelength=2.5, loc='best')

    fig.tight_layout()

    if save_path:
        fig.savefig(
            save_path,
            format=save_format,
            dpi=dpi,
            bbox_inches='tight',
            transparent=transparent,
        )
        print(f'Figure saved as {save_path}.')

    return fig, ax


fig, ax = plot_tpr_curves(
    aa_scores=aa_scores,
    aa_scores_ext=aa_scores_ext,
    thresholds=np.arange(-10, 16, 1),
    width_mode='two_column',
    fig_height=3.0,
    fontsize=7,
    lw=1.0,
    legend_style='inside_bl',
    save_path='tpr_curves.svg',
)

